# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.7 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 2880
Session ID: 8c5f4870-e0a4-4fa0-a560-4e1fdd5937d3
Applying the following default arguments:
--glue_kernel_version 1.0.7
--enable-glue-datacatalog true
Waiting for session 8c5f4870-e0a4-4fa0-a560-4e1fdd5937d3 to get into ready status...
Session 8c5f4870-e0a4-4fa0-a560-4e1fdd5937d3 ha

In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Configuration
S3_BUCKET = "mission-deh-hof-nyc-tlc-030798167757"
RAW_PATH = f"s3://{S3_BUCKET}/raw/hvfhv/"
CURATED_PATH = f"s3://{S3_BUCKET}/curated/hvfhv/"

# Target specific partition for development
TARGET_YEAR = "2024"
TARGET_MONTH = "01"
SPECIFIC_PATH = f"{RAW_PATH}year={TARGET_YEAR}/month={TARGET_MONTH}/"

print(f"Loading data from: {SPECIFIC_PATH}")

# Load raw data
try:
    raw_df = spark.read.parquet(SPECIFIC_PATH)
    print("✅ Data loaded successfully!")
    
    # Basic data exploration
    print(f"\n📊 Dataset Overview:")
    print(f"Total records: {raw_df.count():,}")
    print(f"Total columns: {len(raw_df.columns)}")
    
    # Show schema
    print(f"\n📋 Schema:")
    raw_df.printSchema()
    
    # Show sample data
    print(f"\n🔍 Sample Data (first 5 rows):")
    raw_df.show(5, truncate=False)
    
    # Column analysis
    print(f"\n📈 Column Analysis:")
    for col_name in raw_df.columns:
        null_count = raw_df.filter(col(col_name).isNull()).count()
        null_percentage = (null_count / raw_df.count()) * 100
        print(f"{col_name}: {null_count:,} nulls ({null_percentage:.2f}%)")
    
    # Check unique license numbers
    license_counts = raw_df.groupBy("hvfhs_license_num").count().orderBy(desc("count"))
    print(f"\n🚗 License Distribution:")
    license_counts.show()
    
    print(f"\n✅ Data loading completed! Ready for transformations...")
    
except Exception as e:
    print(f"❌ Error loading data: {str(e)}")
    raise


Loading data from: s3://mission-deh-hof-nyc-tlc-030798167757/raw/hvfhv/year=2024/month=01/
✅ Data loaded successfully!

📊 Dataset Overview:
Total records: 19,663,930
Total columns: 24

📋 Schema:
root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = tru

In [3]:
# Core transformation function
def transform_raw_to_curated(raw_df):
    """
    Transform raw HVFHV data to curated layer per mapping requirements
    """
    
    print("🔄 Starting core transformations...")
    
    # Apply all transformations from mapping sheet
    curated_df = raw_df.select(
        
        # Direct passthrough fields (no transformation)
        col("hvfhs_license_num"),
        col("dispatching_base_num"), 
        col("originating_base_num"),
        
        # Timestamp fields - cast to TIMESTAMP (already correct type)
        col("request_datetime").cast("timestamp").alias("request_datetime"),
        col("on_scene_datetime").cast("timestamp").alias("on_scene_datetime"), 
        col("pickup_datetime").cast("timestamp").alias("pickup_datetime"),
        col("dropoff_datetime").cast("timestamp").alias("dropoff_datetime"),
        
        # Integer fields - cast to INTEGER (already correct type)
        col("PULocationID").cast("integer").alias("PULocationID"),
        col("DOLocationID").cast("integer").alias("DOLocationID"),
        
        # Decimal fields - cast to DECIMAL(10,2) and handle nulls
        coalesce(col("trip_miles"), lit(0.0)).cast("decimal(10,2)").alias("trip_miles"),
        col("trip_time").cast("integer").alias("trip_time_seconds"),  # Rename per mapping
        coalesce(col("base_passenger_fare"), lit(0.0)).cast("decimal(10,2)").alias("base_passenger_fare"),
        coalesce(col("tolls"), lit(0.0)).cast("decimal(10,2)").alias("tolls"),
        coalesce(col("bcf"), lit(0.0)).cast("decimal(10,2)").alias("bcf"),
        coalesce(col("sales_tax"), lit(0.0)).cast("decimal(10,2)").alias("sales_tax"),
        coalesce(col("congestion_surcharge"), lit(0.0)).cast("decimal(10,2)").alias("congestion_surcharge"),
        coalesce(col("airport_fee"), lit(0.0)).cast("decimal(10,2)").alias("airport_fee"),
        coalesce(col("tips"), lit(0.0)).cast("decimal(10,2)").alias("tips"),
        coalesce(col("driver_pay"), lit(0.0)).cast("decimal(10,2)").alias("driver_pay"),
        
        # Boolean flag conversions (Y/N to TRUE/FALSE)
        when(col("shared_request_flag") == "Y", True).otherwise(False).alias("shared_request_flag"),
        when(col("shared_match_flag") == "Y", True).otherwise(False).alias("shared_match_flag"),
        when(col("access_a_ride_flag") == "Y", True).otherwise(False).alias("access_a_ride_flag"),
        when(col("wav_request_flag") == "Y", True).otherwise(False).alias("wav_request_flag"),
        when(col("wav_match_flag") == "Y", True).otherwise(False).alias("wav_match_flag"),
        
        # Missing field - add cbd_congestion_fee (new field starting Jan 5, 2025)
        lit(0.0).cast("decimal(10,2)").alias("cbd_congestion_fee"),
        
        # Derived fields from mapping sheet
        col("pickup_datetime").cast("date").alias("trip_date"),
        hour("pickup_datetime").alias("pickup_hour"),
        
        # Day of week as string - using nested when statements
        when(dayofweek("pickup_datetime") == 1, "Sunday")
        .when(dayofweek("pickup_datetime") == 2, "Monday") 
        .when(dayofweek("pickup_datetime") == 3, "Tuesday")
        .when(dayofweek("pickup_datetime") == 4, "Wednesday")
        .when(dayofweek("pickup_datetime") == 5, "Thursday")
        .when(dayofweek("pickup_datetime") == 6, "Friday")
        .when(dayofweek("pickup_datetime") == 7, "Saturday")
        .alias("day_of_week"),
        
        # Weekend flag
        when(dayofweek("pickup_datetime").isin([1, 7]), True).otherwise(False).alias("is_weekend"),
        
        # Trip duration in minutes
        (col("trip_time") / 60.0).cast("decimal(10,2)").alias("trip_duration_minutes"),
        
        # Total fare calculation
        (coalesce(col("base_passenger_fare"), lit(0.0)) + 
         coalesce(col("tolls"), lit(0.0)) + 
         coalesce(col("bcf"), lit(0.0)) + 
         coalesce(col("sales_tax"), lit(0.0)) + 
         coalesce(col("congestion_surcharge"), lit(0.0)) + 
         coalesce(col("airport_fee"), lit(0.0)) + 
         lit(0.0)  # cbd_congestion_fee
        ).cast("decimal(10,2)").alias("total_fare"),
        
        # Total amount (total_fare + tips)
        (coalesce(col("base_passenger_fare"), lit(0.0)) + 
         coalesce(col("tolls"), lit(0.0)) + 
         coalesce(col("bcf"), lit(0.0)) + 
         coalesce(col("sales_tax"), lit(0.0)) + 
         coalesce(col("congestion_surcharge"), lit(0.0)) + 
         coalesce(col("airport_fee"), lit(0.0)) + 
         lit(0.0) +  # cbd_congestion_fee
         coalesce(col("tips"), lit(0.0))
        ).cast("decimal(10,2)").alias("total_amount"),
        
        # Average speed calculation (handle division by zero)
        when(col("trip_time") > 0, 
             (col("trip_miles") / (col("trip_time") / 3600.0))
        ).otherwise(None).cast("decimal(10,2)").alias("avg_speed_mph"),
        
        # Airport trip flag
        when(coalesce(col("airport_fee"), lit(0.0)) > 0, True).otherwise(False).alias("is_airport_trip"),
        
        # Partition columns
        year("pickup_datetime").alias("year"),
        month("pickup_datetime").alias("month"), 
        dayofmonth("pickup_datetime").alias("day"),
        
        # Processing timestamp
        current_timestamp().alias("processing_timestamp")
    )
    
    print("✅ Core transformations completed!")
    return curated_df

# Apply transformations
curated_df = transform_raw_to_curated(raw_df)

# Show results
print(f"\n📊 Curated Dataset Overview:")
print(f"Total records: {curated_df.count():,}")
print(f"Total columns: {len(curated_df.columns)}")

print(f"\n📋 Curated Schema:")
curated_df.printSchema()

print(f"\n🔍 Sample Transformed Data:")
curated_df.show(5, truncate=False)


🔄 Starting core transformations...
✅ Core transformations completed!

📊 Curated Dataset Overview:
Total records: 19,663,930
Total columns: 38

📋 Curated Schema:
root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp (nullable = true)
 |-- on_scene_datetime: timestamp (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: decimal(10,2) (nullable = true)
 |-- trip_time_seconds: integer (nullable = true)
 |-- base_passenger_fare: decimal(10,2) (nullable = true)
 |-- tolls: decimal(10,2) (nullable = true)
 |-- bcf: decimal(10,2) (nullable = true)
 |-- sales_tax: decimal(10,2) (nullable = true)
 |-- congestion_surcharge: decimal(10,2) (nullable = true)
 |-- airport_fee: decimal(10,2) (null

In [4]:
# Step 3: Data Validation
def validate_curated_data(curated_df):
    print("🔍 Starting data quality validation...")
    
    initial_count = curated_df.count()
    print(f"Initial record count: {initial_count:,}")
    
    # Apply validation filters
    validated_df = curated_df.filter(
        col("pickup_datetime").isNotNull() &
        col("dropoff_datetime").isNotNull() &
        col("PULocationID").isNotNull() &
        col("DOLocationID").isNotNull() &
        (col("trip_miles") >= 0) & (col("trip_miles") <= 500) &
        (col("base_passenger_fare") >= 0) & (col("base_passenger_fare") <= 10000) &
        (col("dropoff_datetime") > col("pickup_datetime")) &
        (col("pickup_datetime") >= col("request_datetime")) &
        col("hvfhs_license_num").isin(['HV0002', 'HV0003', 'HV0004', 'HV0005'])
    )
    
    validated_count = validated_df.count()
    rejected_count = initial_count - validated_count
    rejection_rate = (rejected_count / initial_count) * 100
    
    print(f"✅ Validation completed!")
    print(f"Valid records: {validated_count:,}")
    print(f"Rejected records: {rejected_count:,}")
    print(f"Rejection rate: {rejection_rate:.2f}%")
    
    return validated_df

# Apply validation
validated_df = validate_curated_data(curated_df)

# Show sample validated data
print(f"\n🔍 Sample Validated Data:")
validated_df.show(5, truncate=False)


🔍 Starting data quality validation...
Initial record count: 19,663,930
✅ Validation completed!
Valid records: 19,475,489
Rejected records: 188,441
Rejection rate: 0.96%

🔍 Sample Validated Data:
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+-----------------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+------------------+----------+-----------+-----------+----------+---------------------+----------+------------+-------------+---------------+----+-----+---+--------------------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|request_datetime   |on_scene_datetime  |pickup_datetime    |dropoff_datetime   |PULocationID|DOLocationID|trip_miles|trip_time_seconds|base_passenger_fare|tolls|bcf |sales_tax|congestio

In [6]:
# Step 4: Write Validated Data to S3
import boto3

# Get AWS account ID
account_id = boto3.client('sts').get_caller_identity()['Account']
output_path = f"s3://mission-deh-hof-nyc-tlc-{account_id}/curated/hvfhv/"

print(f"💾 Writing validated data to: {output_path}")

validated_df.write \
    .mode("overwrite") \
    .partitionBy("year", "month", "day") \
    .parquet(output_path)

print("✅ Data successfully written to S3 curated layer!")
print(f"📊 Final count: {validated_df.count():,} records")


💾 Writing validated data to: s3://mission-deh-hof-nyc-tlc-030798167757/curated/hvfhv/
✅ Data successfully written to S3 curated layer!
📊 Final count: 19,475,489 records
